## Commercial Bank of Ethiopia Review Analysis

Import Required Libraries

In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath('C:/Users/dagic/OneDrive/Documents/KAIM/Week_2/fintech-review-analytics'))

print('Path set. Python will now look in:', os.path.abspath('C:/Users/dagic/OneDrive/Documents/KAIM/Week_2/fintech-review-analytics'))

Path set. Python will now look in: C:\Users\dagic\OneDrive\Documents\KAIM\Week_2\fintech-review-analytics


In [2]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime
from src.data_scrapper import scrape_metadata, scrape_reviews, data_quality_check
from src.data_preprocessor import missing_values, duplicate_reviews, check_dateFormat, missing_values, normalize_date, clean_text, invalid_reviews, remove_duplicates, save_cleaned_data, preprocessing_report

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


Defining Variables

In [3]:
App_Id = "com.combanketh.mobilebanking"
bank = 'CBE'
source = 'Google Play Store'
print(f" Scraping Review for Bank: {bank} with App ID: {App_Id} from Source: {source}")


 Scraping Review for Bank: CBE with App ID: com.combanketh.mobilebanking from Source: Google Play Store


## Mobile App Review Data Scraping from Google Play Store

In [4]:
# Get meta data for the app
scrape_metadata(App_Id, bank)

App Info for CBE
App Title   : Commercial Bank of Ethiopia
Current Score: 4.290336
Total Ratings: 48,370
Total Reviews: 9,312
Installs     : 5,000,000+


{'title': 'Commercial Bank of Ethiopia',
 'description': 'Commercial Bank of Ethiopia Mobile Banking\r\n\r\nThe Official app of CBE for Android\r\n\r\nCBE Android Mobile application gives you access to your account on your Android phone. Now, you can perform your banking tasks from the palm of your hand, from anywhere at anytime!\r\n\r\nWhat you can do?\r\n- Real time Account Balance\r\n- Account Statement\r\n- Funds Transfer between own account\r\n- Pay to your beneficiaries\r\n- Manage Beneficiary(Add, List and Delete beneficiaries)\r\n- Exchange Rate\r\n- Local Money Transfer using Mobile number\r\n- ATM Locator and much more.\r\n\r\nOnce you download the application, you can get Authorization code and PIN from your CBE Branch at any time.\r\n\r\nFor further information, please e-mail to us :-   MBandIB@cbe.com.et',
 'descriptionHTML': 'Commercial Bank of Ethiopia Mobile Banking<br><br>The Official app of CBE for Android<br><br>CBE Android Mobile application gives you access to your

In [5]:
#scrape 500 reviews for the app
df_reviews, continuation_token = scrape_reviews(App_Id, bank, count=500)
df_reviews = pd.DataFrame(df_reviews)

Collected 500 raw reviews for CBE app.


In [6]:
print(type(df_reviews))

<class 'pandas.DataFrame'>


In [7]:
print(df_reviews.head())
print(df_reviews.columns)

                               reviewId         userName  \
0  46357e27-661d-4136-bf66-ca7bb91e1427       Muaz ahmed   
1  bfc27568-1471-4937-934f-e5325ea96f46      Alem Melese   
2  9d8ba9a5-4899-4af9-bef1-f12a5dfd7f3a         Sami Fan   
3  07dea887-2fcf-446c-ac8d-753be7c90256      Tes fish Ab   
4  74bf72d1-d6ce-4bd7-9a4d-3950846e5aba  Ahamed Pharmacy   

                                           userImage  \
0  https://play-lh.googleusercontent.com/a-/ALV-U...   
1  https://play-lh.googleusercontent.com/a-/ALV-U...   
2  https://play-lh.googleusercontent.com/a/ACg8oc...   
3  https://play-lh.googleusercontent.com/a/ACg8oc...   
4  https://play-lh.googleusercontent.com/a/ACg8oc...   

                               content  score  thumbsUpCount  \
0                                  wow      4              0   
1                             nice app      5              0   
2                            formative      5              0   
3  best app for financial activities 🙌      5 

In [8]:
# Inspect what a single raw review looks like

print("Keys in a single review:")
print(list(df_reviews.iloc[0].keys()))

print("\nFirst raw review (sample):")
for key, value in df_reviews.iloc[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 46357e27-661d-4136-bf66-ca7bb91e1427
  userName: Muaz ahmed
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjUOdSBGivAJeRB118C15uPeFNOrrvo3Pr8Dmu2tuiHGNwqhm29J
  content: wow
  score: 4
  thumbsUpCount: 0
  reviewCreatedVersion: nan
  at: 2026-05-14 11:52:51
  replyContent: None
  repliedAt: None
  appVersion: nan


In [9]:
# Step 3: Extract only the columns we need
raw_data = []

for r in df_reviews.itertuples():
    raw_data.append({
        'review_id': r.reviewId,
        'review'   : r.content,
        'rating'   : r.score,
        'date'     : r.at,
        'bank'     : 'Awash Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,46357e27-661d-4136-bf66-ca7bb91e1427,wow,4,2026-05-14 11:52:51,Awash Bank,Google Play
1,bfc27568-1471-4937-934f-e5325ea96f46,nice app,5,2026-05-14 11:28:23,Awash Bank,Google Play
2,9d8ba9a5-4899-4af9-bef1-f12a5dfd7f3a,formative,5,2026-05-14 09:49:46,Awash Bank,Google Play
3,07dea887-2fcf-446c-ac8d-753be7c90256,best app for financial activities 🙌,5,2026-05-14 09:46:29,Awash Bank,Google Play
4,74bf72d1-d6ce-4bd7-9a4d-3950846e5aba,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,5,2026-05-14 05:13:13,Awash Bank,Google Play


## Exploring the Raw Data

In [10]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [11]:
print("Rating Distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 4) 
    print(f"{rating} stars: {bar} ({count} reviews)")


Rating Distribution:
5 stars: █████████████████████████████████████████████████████████████████████████████████████ (340 reviews)
4 stars: ██████████ (42 reviews)
3 stars: ████████ (35 reviews)
2 stars: ███ (12 reviews)
1 stars: █████████████████ (71 reviews)


In [12]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-14 11:52:51
1   2026-05-14 11:28:23
2   2026-05-14 09:49:46
3   2026-05-14 09:46:29
4   2026-05-14 05:13:13
5   2026-05-14 03:53:56
6   2026-05-13 22:44:01
7   2026-05-13 13:28:58
8   2026-05-13 10:16:37
9   2026-05-13 09:18:45

Date dtype: datetime64[us]


## Data Quality Audit

In [13]:
data_quality_check(df_raw)

Data Quality Check:
------------------------------
Total reviews collected: 500
Missing values per column:
review_id    0
review       0
rating       0
date         0
bank         0
source       0
dtype: int64


In [14]:
missing_values(df_raw)

Removed 0 rows with missing critical data
Remaining: 500 reviews


,review_id,review,rating,date,bank,source
0,46357e27-661d-4136-bf66-ca7bb91e1427,wow,4,2026-05-14 11:52:51,Awash Bank,Google Play
1,bfc27568-1471-4937-934f-e5325ea96f46,nice app,5,2026-05-14 11:28:23,Awash Bank,Google Play
2,9d8ba9a5-4899-4af9-bef1-f12a5dfd7f3a,formative,5,2026-05-14 09:49:46,Awash Bank,Google Play
3,07dea887-2fcf-446c-ac8d-753be7c90256,best app for financial activities 🙌,5,2026-05-14 09:46:29,Awash Bank,Google Play
4,74bf72d1-d6ce-4bd7-9a4d-3950846e5aba,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,5,2026-05-14 05:13:13,Awash Bank,Google Play
...,...,...,...,...,...,...
495,446d14c5-2a58-49df-8035-dae42208656b,strong 💪,5,2026-03-03 15:11:02,Awash Bank,Google Play
496,91c651cf-14a7-4105-b375-a8187c7aa118,Good,5,2026-03-03 09:42:47,Awash Bank,Google Play
497,27b5e7f8-5e56-425e-921b-8a6ea00fcf9d,Excellent,5,2026-03-03 08:44:53,Awash Bank,Google Play
498,08bff884-5d89-4d4d-9cd5-b2bf78dc8f58,Your app is not working for android version 8....,2,2026-03-03 03:46:13,Awash Bank,Google Play


Check for Duplicates

In [15]:
duplicate_reviews(df_raw)

Duplicate reviews:
------------------------------
Total duplicate review IDs: 0
Total duplicate review texts: 151
Total empty reviews: 0
Total duplicate reviews: 0


In [16]:
# Copying the raw DataFrame to work on a clean version
df = df_raw.copy()
print("Data copied for preprocessing.")

Data copied for preprocessing.


Remove Missing Data

In [17]:
df = missing_values(df)

Removed 0 rows with missing critical data
Remaining: 500 reviews


Remove Duplicates

In [18]:
df = remove_duplicates(df)

Removed 0 duplicate reviews based on review_id
Remaining: 500 reviews


Normalize Date

In [19]:
check_dateFormat(df)
df = normalize_date(df)


Checking date format:
------------------------------
Sample dates: 2026-05-14 11:52:51
Data type of 'date' column: datetime64[us]
  Target format: YYYY-MM-DD (string or date object)
Dates normalized to YYYY-MM-DD format
dtype: str

Date range: 2026-03-03 to 2026-05-14


Clean white spaces

In [20]:
df = clean_text(df)

Remove Invalid Reviews

In [21]:
df = invalid_reviews(df)

Invalid ratings (outside 1–5): 0
Remaining reviews after removing invalid ratings: 500
Data type of 'rating' column: int64


## Cleaned Data

In [ ]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Save Cleaned Data

In [ ]:
df = save_cleaned_data(df, "cbe_reviews")

## Report for Data Preprocessing

In [ ]:
preprocessing_report(df_raw, df)